# 02 — PCP: collect → train Q-corrector → 3-way eval

Predict-Correct-Perturb on LIBERO-PRO. Collection + eval run on a GPU runtime; the
Q-corrector trains in-notebook and logs to Weights & Biases.

## 1. Secrets + install

In [ ]:
import os
from google.colab import userdata
for k in ('SUPABASE_URL', 'SUPABASE_SERVICE_KEY', 'HF_TOKEN', 'WANDB_API_KEY'):
    os.environ[k] = userdata.get(k)
GH_PAT = userdata.get('GH_PAT')
![ -d pnp-vla ] || git clone -q https://$GH_PAT@github.com/ArjunS07/pnp-vla.git
!cd pnp-vla && git pull -q && pip install -q -e '.[sim]'

## 2. Environment + model + store

In [ ]:
from pnp.env_setup import setup_environment
setup_environment()

In [ ]:
from pnp import libero_env, libero_pro, models
from pnp.store import SupabaseStore
from pnp.config import PCPConfig

libero_env.init_libero_benchmark()
policy, preprocess, postprocess = models.load_pi05()
device = models.default_device()
store = SupabaseStore()
cfg = PCPConfig()
# pro_eps = libero_pro.build_libero_pro_episodes(..., episode_idxs=range(10))

## 3. Collect labeled (z_hat, obs_enc) chunks → qc_rollouts

In [ ]:
from pnp.rollout import pcp_collect
# pcp_collect(store, policy, preprocess, device, pro_eps, cfg=cfg, experiment='pcp-v1')

## 4. Train + calibrate the Q-corrector (wandb) → q_correctors

In [ ]:
import wandb
from pnp.pcp import load_qc_samples, train_q_corrector, ckpt_bytes, new_q_ckpt_id

qc_rows = store.load_qc_rows(experiment='pcp-v1')
samples = load_qc_samples(qc_rows, cfg)
run = wandb.init(project='pnp-qcorrector', name='pcp-v1', config={'experiment': 'pcp-v1'})
q_model, q_scaler, meta, split_ids = train_q_corrector(
    samples, device, cfg=cfg, wandb_run=run, experiment='pcp-v1')
q_ckpt_id = new_q_ckpt_id()
store.register_q_corrector(q_ckpt_id, ckpt_bytes(q_model, q_scaler, meta), meta,
                           split_ids=split_ids)
run.log_artifact  # checkpoint is registered in Supabase under q_ckpt_id
run.finish()
print('q_ckpt_id =', q_ckpt_id, ' val_auc =', meta['val_auc'])

## 5. 3-way eval (vanilla / pnp-only / pcp) on hard tasks → qc_eval

In [ ]:
from pnp.rollout import pcp_eval
# pcp_eval(store, policy, preprocess, device, pro_eps, q_ckpt_id, cfg=cfg, experiment='pcp-v1')